# Capstone — Content Refresh Priority on the FlyRank Warehouse

**Lane:** Refresh / Content Opportunity Scoring
**Question:** Which pages should a content team review first, ranked by the risk that they lose search visibility in the coming month?

This notebook mirrors the deployed research paper. It runs top to bottom and
produces every number, chart, and table the paper cites — so the paper never
contains a figure this notebook can't regenerate.

**Before you publish:** run `Run all` once end to end, then copy the numbers it
prints into the paper. Anywhere you see `[FROM RUN]` below, a real number goes in.
Do not write a number into the paper that this notebook didn't print.

**Seeds:** `RANDOM_STATE = 42`.

## 1. Question

**The decision this supports.** A content team has bandwidth to review a handful
of pages per week out of tens of thousands. The decision is: *which pages go into
this month's review queue?*

**Unit of analysis.** One content item (one page) belonging to one client, observed
over a 120-day window of daily search-performance data.

**Output.** A ranked queue with a reason code per page — not a yes/no verdict.

**The action a human takes.** An editor opens the top N pages, checks them against
the reason code, and decides whether to refresh, re-title, or leave alone. Nothing
is automated.

**Cost of a wrong call.** A false positive burns editor hours on a page that was
fine. A false negative is the more expensive error: a page with real traffic keeps
sliding, quietly, and the loss compounds before anyone notices. The ranking is
tuned accordingly — precision at the top of the list is what matters, because that
is the only part anyone reads.

**Why ML rather than a rule.** A fixed rule has to commit to thresholds in advance.
This notebook tests that assumption directly by scoring a transparent rule and a
learned model on the same split, and reports the gap honestly — including when the
gap is small.

In [1]:
%pip -q install duckdb huggingface_hub scikit-learn matplotlib
%pip install -q --upgrade duckdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 45.9 MB/s eta 0:00:00


In [2]:
import os, json, getpass, textwrap
import numpy as np
import pandas as pd
import duckdb
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"   # build v20260703
TABLES = {
    "dim_clients":  f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content":  f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily":   f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_query90": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}
DAILY = TABLES["fact_daily"]

# ---------------------------------------------------------------------------
# ANCHOR DATE -- the single most important switch in this notebook.
#
# Every window is measured backwards from ANCHOR. The label lives in
# (ANCHOR-30, ANCHOR].
#
# The panel runs 2025-01-27 -> 2026-06-30. The FINAL month is the sealed test
# month: if label logic is developed against it, the development happens inside
# the test window. So iterate on a mid-panel anchor, and run the sealed anchor
# exactly once, when the query is final.
#
# NOTE: fact_content_daily_performance_sample is the final month ONLY. It is
# fine for checking query mechanics and useless -- actively misleading -- for
# label logic, because a 120-day lookback from it has no history to read.
# It is deliberately not wired in below.
# ---------------------------------------------------------------------------
DEV_ANCHOR    = "2026-03-31"   # mid-panel: develop here
SEALED_ANCHOR = "2026-06-30"   # final month: run ONCE, for the published figures

ANCHOR = DEV_ANCHOR            # <-- switch to SEALED_ANCHOR for the published run
SEALED_RUN = (ANCHOR == SEALED_ANCHOR)

print(f"anchor date: {ANCHOR}" + ("   [SEALED RUN -- publish these numbers]" if SEALED_RUN
                                  else "   [development run -- do not publish]"))

anchor date: 2026-03-31   [development run -- do not publish]


## 2. Data

**Release.** `FlyRank/internship-warehouse`, build `v20260703` (gated Hugging Face
release), read through DuckDB over `hf://` so no full download occurs. The panel runs
**2025-01-27 to 2026-06-30** — roughly 17 months, 78,835,655 daily rows, 519,606 content
items across 104 pseudonymized clients. Section 2's first cell verifies those counts
against the release before anything downstream runs.

**Tables used.**
- `fact_content_daily_performance` — daily impressions, clicks, and average position per content item. The only table the model reads.
- `dim_clients` / `dim_content` — used for eligibility and counts, never for features.

**One table deliberately left out.** `fact_content_query_90d` carries appealing query-mix
signals, but it aggregates a **fixed 90-day window** that overlaps the label window. Only
`*_prev30`-style columns from it would be window-safe, so the table is excluded entirely
rather than joined and hoped over.

**The anchor date, and why the final month is not the development set.**
All windows are measured backwards from a single `ANCHOR`. The final month of the panel
(June 2026) is the natural outcome window of any past-to-future label, so it is treated as
a **sealed test month**: label logic was developed against a mid-panel anchor
(`2026-03-31`) and the published figures come from a single run at the sealed anchor
(`2026-06-30`). Developing against the final month would mean tuning inside the test window.

For the same reason `fact_content_daily_performance_sample` is never used here. It contains
the final month only, so a 120-day lookback from it has no history to read — it is useful
for checking query mechanics and actively misleading for label logic.

**Exclusions, and what each one protects against.**
- Clients whose data does not span the full 120-day window ending at the anchor. History depth varies sharply per client; without this filter, a client whose feed stops mid-window shows a collapse in the label window that is *missing data*, not decline.
- Content items with fewer than 100 impressions in W1. Below that floor a 20% swing is noise.
- Content items with fewer than 25 reporting days inside the label window — same failure mode as the first exclusion, at the page level.

**Two data traps handled explicitly.**
- `gsc_avg_position = 0` means *no data*, not rank zero. Averaging those zeros in would pull every position toward "rank 1", i.e. toward looking excellent. Only positive values are averaged, and the count of positive days is retained.
- Missing position is flagged (`has_position`) rather than silently filled, because missingness in this warehouse follows content type — a blind fill injects a category signal disguised as a measurement.

**Cost discipline.** The 79M-row scan runs once per anchor and caches its aggregate to
`work/outputs/`; repeated full scans hit HTTP 429 rate limits.

**Public safety.** Only pseudonymous `client_hash_id` and `content_hash_id` appear, used
strictly for grouping — never as features, never printed into the paper. No product flags
or health scores are used as features either: those encode a decision someone already made,
which makes them a baseline to beat rather than an input.

In [3]:
# --- verify the release matches the documented build before trusting anything ---
shape = con.sql(f"""
    SELECT COUNT(*) AS rows,
           MIN(report_date) AS first_day,
           MAX(report_date) AS last_day,
           COUNT(DISTINCT client_hash_id) AS clients
    FROM {DAILY}
""").df()
display(shape)
print("expected (build v20260703): 78,835,655 rows, 2025-01-27 -> 2026-06-30\n")

# --- grain probe: one row per report_date x client x content, or the SQL below lies ---
dupes = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM {DAILY}
    WHERE report_date BETWEEN DATE '{ANCHOR}' - INTERVAL 7 DAY AND DATE '{ANCHOR}'
    GROUP BY 1,2,3 HAVING COUNT(*) > 1 LIMIT 5
""").df()
assert len(dupes) == 0, f"grain is not what we assumed -- duplicates found:\n{dupes}"
print("grain probe: clean (one row per date x client x content)")

# --- schema check: confirm the column names this notebook assumes ---
display(con.sql(f"DESCRIBE SELECT * FROM {DAILY} LIMIT 1").df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows,first_day,last_day,clients
0,78835655,2025-01-27,2026-06-30,70


expected (build v20260703): 78,835,655 rows, 2025-01-27 -> 2026-06-30



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

grain probe: clean (one row per date x client x content)


,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [4]:
CACHE = f"work/outputs/panel_{ANCHOR}.parquet"

# The full scan is expensive and rate-limited (HTTP 429 on repeats). Run it once
# per anchor, cache the small aggregate, and re-read the cache thereafter.
if os.path.exists(CACHE):
    panel = pd.read_parquet(CACHE)
    print(f"loaded cached panel for anchor {ANCHOR}: {len(panel):,} rows")
else:
    feat_sql = f"""
    WITH eligible_clients AS (
        -- A client only qualifies if its data actually reaches the anchor.
        -- Without this, a client whose feed stops mid-window shows a collapse in
        -- L that is missing data, not decline -- the label would be manufactured.
        SELECT client_hash_id
        FROM {DAILY}
        GROUP BY 1
        HAVING MAX(report_date) >= DATE '{ANCHOR}' - INTERVAL 2 DAY
           AND MIN(report_date) <= DATE '{ANCHOR}' - INTERVAL 120 DAY
    ),
    blocks AS (
        SELECT
            f.client_hash_id,
            f.content_hash_id,

            -- W1: most recent FEATURE block (anchor-60, anchor-30]
            SUM(CASE WHEN f.report_date >  DATE '{ANCHOR}' - INTERVAL 60 DAY
                      AND f.report_date <= DATE '{ANCHOR}' - INTERVAL 30 DAY
                     THEN f.gsc_impressions ELSE 0 END) AS imp_w1,
            SUM(CASE WHEN f.report_date >  DATE '{ANCHOR}' - INTERVAL 60 DAY
                      AND f.report_date <= DATE '{ANCHOR}' - INTERVAL 30 DAY
                     THEN f.gsc_clicks ELSE 0 END) AS clk_w1,
            -- gsc_avg_position = 0 means NO DATA, not rank zero. Averaging the
            -- zeros in would drag every position toward "rank 1" -- i.e. toward
            -- looking excellent. Only positive values are averaged.
            AVG(CASE WHEN f.report_date >  DATE '{ANCHOR}' - INTERVAL 60 DAY
                      AND f.report_date <= DATE '{ANCHOR}' - INTERVAL 30 DAY
                      AND f.gsc_avg_position > 0
                     THEN f.gsc_avg_position END) AS pos_w1,
            STDDEV_SAMP(CASE WHEN f.report_date >  DATE '{ANCHOR}' - INTERVAL 60 DAY
                              AND f.report_date <= DATE '{ANCHOR}' - INTERVAL 30 DAY
                              AND f.gsc_avg_position > 0
                             THEN f.gsc_avg_position END) AS pos_sd_w1,
            COUNT(CASE WHEN f.report_date >  DATE '{ANCHOR}' - INTERVAL 60 DAY
                        AND f.report_date <= DATE '{ANCHOR}' - INTERVAL 30 DAY
                        AND f.gsc_avg_position > 0
                       THEN 1 END) AS pos_days_w1,
            COUNT(DISTINCT CASE WHEN f.report_date >  DATE '{ANCHOR}' - INTERVAL 60 DAY
                                 AND f.report_date <= DATE '{ANCHOR}' - INTERVAL 30 DAY
                                 AND f.gsc_impressions > 0
                                THEN f.report_date END) AS active_days_w1,

            -- W2: (anchor-90, anchor-60]
            SUM(CASE WHEN f.report_date >  DATE '{ANCHOR}' - INTERVAL 90 DAY
                      AND f.report_date <= DATE '{ANCHOR}' - INTERVAL 60 DAY
                     THEN f.gsc_impressions ELSE 0 END) AS imp_w2,
            SUM(CASE WHEN f.report_date >  DATE '{ANCHOR}' - INTERVAL 90 DAY
                      AND f.report_date <= DATE '{ANCHOR}' - INTERVAL 60 DAY
                     THEN f.gsc_clicks ELSE 0 END) AS clk_w2,
            AVG(CASE WHEN f.report_date >  DATE '{ANCHOR}' - INTERVAL 90 DAY
                      AND f.report_date <= DATE '{ANCHOR}' - INTERVAL 60 DAY
                      AND f.gsc_avg_position > 0
                     THEN f.gsc_avg_position END) AS pos_w2,

            -- W3: (anchor-120, anchor-90]
            SUM(CASE WHEN f.report_date >  DATE '{ANCHOR}' - INTERVAL 120 DAY
                      AND f.report_date <= DATE '{ANCHOR}' - INTERVAL 90 DAY
                     THEN f.gsc_impressions ELSE 0 END) AS imp_w3,

            -- L: LABEL block (anchor-30, anchor] -- never read by a feature
            SUM(CASE WHEN f.report_date >  DATE '{ANCHOR}' - INTERVAL 30 DAY
                      AND f.report_date <= DATE '{ANCHOR}'
                     THEN f.gsc_impressions ELSE 0 END) AS imp_label,
            COUNT(DISTINCT CASE WHEN f.report_date >  DATE '{ANCHOR}' - INTERVAL 30 DAY
                                 AND f.report_date <= DATE '{ANCHOR}'
                                THEN f.report_date END) AS label_days_present
        FROM {DAILY} f
        JOIN eligible_clients e USING (client_hash_id)
        WHERE f.report_date >  DATE '{ANCHOR}' - INTERVAL 120 DAY
          AND f.report_date <= DATE '{ANCHOR}'
        GROUP BY 1, 2
        HAVING imp_w1 >= 100
           AND label_days_present >= 25   -- a near-complete label window, or the
                                          -- "decline" is just absent reporting days
    )
    SELECT * FROM blocks
    """
    panel = con.sql(feat_sql).df()
    panel.to_parquet(CACHE, index=False)
    print(f"scanned and cached: {len(panel):,} rows -> {CACHE}")

n_content_total = con.sql(f"SELECT COUNT(*) FROM {TABLES['dim_content']}").fetchone()[0]
print(f"content items in release:       {n_content_total:,}")
print(f"content items after exclusions: {len(panel):,}")
panel.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

scanned and cached: 70,557 rows -> work/outputs/panel_2026-03-31.parquet
content items in release:       519,606
content items after exclusions: 70,557


,client_hash_id,content_hash_id,imp_w1,clk_w1,pos_w1,pos_sd_w1,pos_days_w1,active_days_w1,imp_w2,clk_w2,pos_w2,imp_w3,imp_label,label_days_present
0,client_3ffa76342f366962,content_9b118b3509d3d54c,135.0,5.0,3.953520,1.488754,26,27,134.0,6.0,3.644581,54.0,110.0,29
1,client_3ffa76342f366962,content_cae1d5374958a649,101.0,0.0,6.450861,2.444493,24,28,72.0,0.0,7.066374,0.0,40.0,30
2,client_3ffa76342f366962,content_dd66eecf9626cab8,265.0,0.0,6.351742,1.705478,30,30,146.0,1.0,7.213658,31.0,240.0,30
3,client_3ffa76342f366962,content_456ab2db28595187,103.0,2.0,4.342622,1.851422,27,27,100.0,3.0,5.055778,61.0,48.0,30
4,client_3ffa76342f366962,content_5573434837db89c5,200.0,6.0,6.865873,1.379616,24,25,16.0,0.0,8.750000,0.0,88.0,29


## 3. Methodology

### Assumptions
1. A page's near-future visibility is partly predictable from its own recent trajectory.
2. Impressions are a usable proxy for visibility; clicks alone confound ranking with snippet quality.
3. Behaviour learned on one set of clients transfers to an unseen client — tested directly by the split, not assumed.

### Label definition
> A page is **declining** if its impressions in the final 30-day window fall below
> 80% of its impressions in the preceding 30-day window.

This is a **forward** label. An earlier iteration of this project used a trailing
`trend_direction` flag computed over the same window as the features, which makes
the task close to circular — the model detects a trend from the metrics that
produced it, rather than predicting anything.

### Window design

```
 |<-- W3 -->|<-- W2 -->|<-- W1 -->|<--  L  -->|
end-120    end-90     end-60     end-30      end
 \________ FEATURES ________/     \_ LABEL _/
```

No feature reads a single day inside `L`.

### Baseline
Two transparent rules, both scored on the same split as the model:
- **Momentum** — rank by how much impressions already fell from W2 to W1. Aimed at the same construct as the label.
- **Opportunity** — `impressions x (1 - CTR)`, the classic click-gap heuristic. Included deliberately to test whether click-opportunity and decline-risk are the same thing. They are not, and the paper reports that as a finding.

### Validation design
`GroupShuffleSplit` on `client_hash_id`, 25% held out. No client's pages appear on
both sides, so the reported score is performance on a **brand-new client** — which
is how the tool would actually be deployed. Time separation is already built into
the label, so the evaluation is both grouped and forward-looking.

### Leakage checks
Three assertions that fail loudly rather than printing reassurance — run below.

In [5]:
d = panel.copy()
eps = 1e-9

# Missingness is informative here, so it gets a flag rather than a silent fill.
# A blind fillna() would inject whatever pattern drives the missingness into the
# feature as if it were signal.
d["has_position"] = d["pos_w1"].notna().astype(int)
pos_med = d["pos_w1"].median()
d["pos_w1"] = d["pos_w1"].fillna(pos_med)
d["pos_w2"] = d["pos_w2"].fillna(pos_med)
d["pos_sd_w1"] = d["pos_sd_w1"].fillna(0.0)

# CTR is computed from raw clicks/impressions, so it is a true 0-1 fraction.
# (The starter CSV's ctr column is a x100 percentage -- 0.76 means 0.76%. That
# trap does not apply to these warehouse counts, but it does apply to any
# comparison against the CSV pipeline.)
d["imp_w1_log"]     = np.log1p(d["imp_w1"])
d["ctr_w1"]         = d["clk_w1"] / (d["imp_w1"] + eps)
d["ctr_w2"]         = d["clk_w2"] / (d["imp_w2"] + eps)
d["ctr_delta"]      = d["ctr_w1"] - d["ctr_w2"]
d["imp_momentum"]   = d["imp_w1"] / (d["imp_w2"] + eps)
d["imp_momentum_2"] = d["imp_w2"] / (d["imp_w3"] + eps)
d["pos_delta"]      = d["pos_w1"] - d["pos_w2"]
d["active_share"]   = d["active_days_w1"] / 30.0

FEATURES = ["imp_w1_log", "ctr_w1", "ctr_delta", "imp_momentum", "imp_momentum_2",
            "pos_w1", "pos_delta", "pos_sd_w1", "active_share", "has_position"]

d["is_declining"] = (d["imp_label"] < 0.8 * d["imp_w1"]).astype(int)

base_rate = float(d["is_declining"].mean())
print(f"anchor: {ANCHOR}   rows modelled: {len(d):,}")
print(f"BASE RATE (declining in the forward window): {base_rate:.1%}")
print(f"majority-class accuracy floor: {max(base_rate, 1-base_rate):.1%}")
print("\nEvery Precision@K in this notebook is reported against that base rate.")

anchor: 2026-03-31   rows modelled: 70,557
BASE RATE (declining in the forward window): 24.8%
majority-class accuracy floor: 75.2%

Every Precision@K in this notebook is reported against that base rate.


In [6]:
# --- LEAKAGE CHECKS (assertions, not messages) ---

# 1. No label-derived column reached the feature list.
banned = {"imp_label", "is_declining", "trend_direction", "trend_pct"}
assert not (banned & set(FEATURES)), f"label-derived feature present: {banned & set(FEATURES)}"

# 2. No feature is the label in disguise.
corr = d[FEATURES].corrwith(d["is_declining"]).abs().sort_values(ascending=False)
display(corr.to_frame("abs_corr_with_label").round(3))
assert corr.max() < 0.95, f"suspiciously high correlation -- inspect {corr.idxmax()}"

# 3. Query-mix features excluded by design.
QUERY_EXCLUSION_NOTE = (
    "fact_content_query_90d aggregates a 90-day window that overlaps the 30-day "
    "label window (end-30, end]. Joining it would leak outcome-period impressions "
    "into the feature set, so it is excluded."
)
print("PASS -- leakage checks cleared.")
print(textwrap.fill(QUERY_EXCLUSION_NOTE, 88))

,abs_corr_with_label
pos_delta,0.069
pos_w1,0.067
ctr_w1,0.051
imp_momentum,0.041
imp_w1_log,0.037
active_share,0.027
imp_momentum_2,0.020
ctr_delta,0.020
has_position,0.007
pos_sd_w1,0.003


PASS -- leakage checks cleared.
fact_content_query_90d aggregates a 90-day window that overlaps the 30-day label window
(end-30, end]. Joining it would leak outcome-period impressions into the feature set, so
it is excluded.


### Ruling out the look-alikes

Not every impression drop is decline. Before a page is called declining, three
rival explanations need to be on the record — the lane guide names them and a
paper that ignores them is claiming more than it measured.

| Rival explanation | What it would mean | Check run below |
|---|---|---|
| **Consolidation** | A sibling page on the same site absorbed this page's demand | Group by `keyword_hash_id`: did the group hold steady while the page fell? |
| **Noise** | A low-volume wiggle with no lasting pattern | The 100-impression floor, plus the persistence split below |
| **Blip vs sustained** | One bad fortnight, not a trend | Label window split in half; strict label requires both halves down |

These are reported as *quantified limitations*, not removed from the data. The
primary label stays as defined; the strict label is a robustness check, and the
paper reports how much the headline result moves between them.

In [7]:
# --- rival explanation 1: persistence (blip vs sustained) ---
# Split the label window in half. A page that fell in both halves is declining in a
# stronger sense than one that had a single bad fortnight.
persist_sql = f"""
    SELECT client_hash_id, content_hash_id,
           SUM(CASE WHEN report_date >  DATE '{ANCHOR}' - INTERVAL 30 DAY
                     AND report_date <= DATE '{ANCHOR}' - INTERVAL 15 DAY
                    THEN gsc_impressions ELSE 0 END) AS imp_l_first,
           SUM(CASE WHEN report_date >  DATE '{ANCHOR}' - INTERVAL 15 DAY
                     AND report_date <= DATE '{ANCHOR}'
                    THEN gsc_impressions ELSE 0 END) AS imp_l_second
    FROM {DAILY}
    WHERE report_date > DATE '{ANCHOR}' - INTERVAL 30 DAY
      AND report_date <= DATE '{ANCHOR}'
    GROUP BY 1, 2
"""
halves = con.sql(persist_sql).df()
d = d.merge(halves, on=["client_hash_id", "content_hash_id"], how="left")

half_w1 = d["imp_w1"] / 2.0
d["is_declining_strict"] = (
    (d["is_declining"] == 1)
    & (d["imp_l_first"]  < 0.8 * half_w1)
    & (d["imp_l_second"] < 0.8 * half_w1)
).astype(int)

strict_rate = float(d["is_declining_strict"].mean())
share_persisted = (d.loc[d["is_declining"] == 1, "is_declining_strict"].mean()
                   if d["is_declining"].sum() else float("nan"))

print(f"primary label (any 20% drop):          {base_rate:.1%}")
print(f"strict label (drop in BOTH halves):    {strict_rate:.1%}")
print(f"share of primary declines that persisted: {share_persisted:.1%}")
print("\nThe gap between these two numbers is how much of the primary label is blip.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

primary label (any 20% drop):          24.8%
strict label (drop in BOTH halves):    16.5%
share of primary declines that persisted: 66.6%

The gap between these two numbers is how much of the primary label is blip.


In [8]:
# --- rival explanation 2: consolidation (a sibling page absorbed the demand) ---
# dim_content carries keyword_hash_id for exactly this: grouping related pages
# without ever revealing what the keyword was.
try:
    kmap = con.sql(f"""
        SELECT content_hash_id, keyword_hash_id
        FROM {TABLES['dim_content']}
        WHERE keyword_hash_id IS NOT NULL
    """).df()
    dc = d.merge(kmap, on="content_hash_id", how="left")

    grp = (dc.dropna(subset=["keyword_hash_id"])
             .groupby(["client_hash_id", "keyword_hash_id"])
             .agg(grp_w1=("imp_w1", "sum"), grp_label=("imp_label", "sum"), n=("imp_w1", "size"))
             .reset_index())
    grp["group_held"] = (grp["grp_label"] >= 0.95 * grp["grp_w1"]).astype(int)

    dc = dc.merge(grp[["client_hash_id", "keyword_hash_id", "group_held", "n"]],
                  on=["client_hash_id", "keyword_hash_id"], how="left")

    # A declining page whose keyword group held steady, alongside siblings, is a
    # consolidation candidate: the demand may have moved rather than vanished.
    dc["consolidation_candidate"] = (
        (dc["is_declining"] == 1) & (dc["group_held"] == 1) & (dc["n"] > 1)
    ).astype(int)

    n_dec = int((dc["is_declining"] == 1).sum())
    n_con = int(dc["consolidation_candidate"].sum())
    print(f"declining pages: {n_dec:,}")
    print(f"of those, in a keyword group that held steady (possible consolidation): "
          f"{n_con:,} ({n_con / max(n_dec, 1):.1%})")
    print("\nThis share is a LIMITATION to state in the paper, not a filter to apply "
          "silently. Report it.")
    CONSOLIDATION_SHARE = n_con / max(n_dec, 1)
except Exception as e:
    print("keyword grouping unavailable in this release build:", e)
    print("State in Limitations that the consolidation check could not be run.")
    CONSOLIDATION_SHARE = None

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

declining pages: 17,484
of those, in a keyword group that held steady (possible consolidation): 0 (0.0%)

This share is a LIMITATION to state in the paper, not a filter to apply silently. Report it.


In [9]:
from sklearn.model_selection import GroupShuffleSplit

X = d[FEATURES]
y = d["is_declining"].values
groups = d["client_hash_id"].values

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups))

X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
y_tr, y_te = y[train_idx], y[test_idx]
d_te = d.iloc[test_idx].copy()

overlap = len(set(d.iloc[train_idx]["client_hash_id"]) & set(d_te["client_hash_id"]))
assert overlap == 0, f"client overlap between train and test: {overlap}"

te_base = float(y_te.mean())
print(f"train: {len(train_idx):,} rows / {d.iloc[train_idx]['client_hash_id'].nunique()} clients")
print(f"test:  {len(test_idx):,} rows / {d_te['client_hash_id'].nunique()} clients")
print(f"client overlap: {overlap} (asserted 0)")
print(f"test-set base rate: {te_base:.1%}")

train: 28,848 rows / 19 clients
test:  41,709 rows / 7 clients
client overlap: 0 (asserted 0)
test-set base rate: 25.2%


## 4. Results (vs baseline)

The honest table: both baselines and the model, on the identical held-out split,
with the base rate printed alongside so no score can be read out of context.

`lift = Precision@K / base_rate`. A lift of 1.00 is indistinguishable from ranking
at random. **Below 1.00 is worse than random**, which is a real and reportable outcome.

In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# Baselines
d_te["baseline_momentum"]    = -d_te["imp_momentum"]
d_te["baseline_opportunity"] = d_te["imp_w1"] * (1 - d_te["ctr_w1"])

# Model
rf = RandomForestClassifier(
    n_estimators=300, max_depth=8, min_samples_leaf=20,
    class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1,
).fit(X_tr, y_tr)
d_te["ml_score"] = rf.predict_proba(X_te)[:, 1]


def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(labels)[order[:k]].mean())


rows = []
for name, score in [
    ("Baseline A - momentum",    d_te["baseline_momentum"]),
    ("Baseline B - opportunity", d_te["baseline_opportunity"]),
    ("Random forest",            d_te["ml_score"]),
]:
    p20, p50 = precision_at_k(score, y_te, 20), precision_at_k(score, y_te, 50)
    rows.append({
        "method": name,
        "P@20": round(p20, 3), "lift@20": round(p20 / te_base, 2),
        "P@50": round(p50, 3), "lift@50": round(p50 / te_base, 2),
        "AUC":  round(float(roc_auc_score(y_te, score)), 3),
    })

results = pd.DataFrame(rows)
print(f"TEST-SET BASE RATE: {te_base:.1%}  <-- a random ranker scores this at every K\n")
display(results)

TEST-SET BASE RATE: 25.2%  <-- a random ranker scores this at every K



,method,P@20,lift@20,P@50,lift@50,AUC
0,Baseline A - momentum,0.25,0.99,0.30,1.19,0.494
1,Baseline B - opportunity,0.40,1.58,0.34,1.35,0.486
2,Random forest,0.40,1.58,0.46,1.82,0.629


In [11]:
importance = (pd.DataFrame({"feature": FEATURES, "importance": rf.feature_importances_})
                .sort_values("importance", ascending=False)
                .reset_index(drop=True))
display(importance.round(4))

print("\nRead this as: which signals the model leans on -- NOT as causes of decline.")

,feature,importance
0,pos_w1,0.1687
1,imp_momentum,0.1583
2,imp_momentum_2,0.1440
3,pos_delta,0.1381
4,imp_w1_log,0.1105
5,pos_sd_w1,0.1009
6,ctr_w1,0.0731
7,ctr_delta,0.0643
8,active_share,0.0420
9,has_position,0.0000



Read this as: which signals the model leans on -- NOT as causes of decline.


### Error analysis

What the model gets wrong matters more than the headline number. Below: the false
positives inside the top 50, profiled against the true positives, so the paper can
say something specific about the failure mode rather than "some errors occurred".

In [12]:
top50 = d_te.nlargest(50, "ml_score")
fp = top50[top50["is_declining"] == 0]
tp = top50[top50["is_declining"] == 1]

print(f"False positives in the top 50: {len(fp)} of 50\n")
profile = pd.DataFrame({
    "false_positive_median": fp[FEATURES].median(),
    "true_positive_median":  tp[FEATURES].median(),
})
profile["gap"] = profile["false_positive_median"] - profile["true_positive_median"]
display(profile.round(3).sort_values("gap", key=abs, ascending=False))

False positives in the top 50: 27 of 50



,false_positive_median,true_positive_median,gap
pos_delta,0.661,0.257,0.404
imp_momentum_2,2.105,2.466,-0.361
pos_sd_w1,1.105,1.378,-0.272
pos_w1,5.523,5.453,0.069
imp_momentum,2.921,2.862,0.059
imp_w1_log,8.935,8.967,-0.032
ctr_delta,-0.001,-0.000,-0.001
ctr_w1,0.000,0.000,0.000
active_share,1.000,1.000,0.000
has_position,1.000,1.000,0.000


## 5. Limitations

**What this work cannot claim.**

1. **No causal claim.** Nothing here is an experiment. The paper reports association
   within a measured window. It cannot say staleness *causes* decline, that a refresh
   *will* reverse a decline, or anything about why a ranking moved.

2. **Nothing about Google's algorithm.** The data records outcomes, not mechanism. Any
   sentence resembling "the algorithm rewards X" is out of scope.

3. **The label is a proxy.** A >20% impression drop is a reasonable stand-in for
   "worth reviewing," but a page can decline for reasons no refresh addresses —
   seasonality, a SERP-layout change, a competitor's launch, or genuinely obsolete demand.

4. **One forward window, one cut.** Results come from a single 30-day forward window
   at the end of the panel. A different month could behave differently; this is not a
   multi-period backtest.

5. **No semantic context.** The model never reads the page. It cannot distinguish a
   stale guide that needs rewriting from an evergreen reference that is fine, or from a
   navigational page that should never be touched.

6. **Unbalanced panel.** History depth varies per client, so pages from
   shorter-history clients are underrepresented among those surviving the 120-day filter.

7. **Survivorship in the exclusion.** Filtering to >=100 impressions removes exactly the
   long tail where some decline happens. The result generalises to visible pages only.

8. **Thresholds are choices.** The 20% drop, the 100-impression floor, and the 30-day
   windows are defensible but arbitrary. Sensitivity to them is untested here.

Language used throughout: **observed, measured, directional, decision-support.**

## 6. Ranked recommendations

The action playbook. The **model supplies the ranking**; the reason codes are a
transparent rule layered on top to explain *why* a page surfaced. That division is
stated explicitly rather than implied — the model does not choose actions.

**No-go list.** No automated deletion or unpublishing; no generative rewriting of
flagged pages without a subject-matter editor; legal, privacy, and compliance pages
are exempt from the queue entirely.

In [13]:
queue = d_te[["client_hash_id", "content_hash_id", "ml_score", "imp_w1",
              "ctr_w1", "pos_w1", "pos_delta", "imp_momentum", "is_declining"]].copy()
queue = queue.sort_values("ml_score", ascending=False).reset_index(drop=True)

ctr_med, imp_med = queue["ctr_w1"].median(), queue["imp_w1"].median()

def reason_code(r):
    if r["pos_delta"] > 1.0:
        return ("Investigate ranking loss", "POSITION_SLIP")
    if r["imp_momentum"] < 0.9:
        return ("Full content refresh", "MOMENTUM_DECAY")
    if r["ctr_w1"] < ctr_med and r["imp_w1"] > imp_med:
        return ("Title / meta optimization", "VISIBLE_LOW_CTR")
    return ("Monitor & hold", "STABLE")

queue[["action", "reason_code"]] = queue.apply(reason_code, axis=1, result_type="expand")

print("Action mix across the full held-out queue:")
display(queue["action"].value_counts().to_frame("pages"))

print("\nHow each reason code actually performs (precision within the code):")
display(queue.groupby("reason_code")["is_declining"]
             .agg(["mean", "count"])
             .rename(columns={"mean": "precision", "count": "pages"})
             .round(3)
             .sort_values("pages", ascending=False))
print(f"\nCompare each against the test-set base rate of {te_base:.1%}.")

Action mix across the full held-out queue:


,pages
action,
Monitor & hold,23442
Investigate ranking loss,7767
Title / meta optimization,5288
Full content refresh,5212



How each reason code actually performs (precision within the code):


,precision,pages
reason_code,,
STABLE,0.240,23442
POSITION_SLIP,0.229,7767
VISIBLE_LOW_CTR,0.321,5288
MOMENTUM_DECAY,0.276,5212



Compare each against the test-set base rate of 25.2%.


In [14]:
# Paper-safe excerpt: aggregate rows only, no pseudonymous IDs.
excerpt = (queue.head(200)
           .groupby("reason_code")
           .agg(pages=("ml_score", "size"),
                median_score=("ml_score", "median"),
                median_impressions=("imp_w1", "median"),
                median_position=("pos_w1", "median"))
           .round(3)
           .sort_values("pages", ascending=False))

print("Top-200 queue, summarised by reason code -- THIS is what goes in the paper:")
display(excerpt)
print("\n(Do not paste content_hash_id or client_hash_id into the published paper.)")

Top-200 queue, summarised by reason code -- THIS is what goes in the paper:


,pages,median_score,median_impressions,median_position
reason_code,,,,
VISIBLE_LOW_CTR,128,0.710,7146.5,4.416
POSITION_SLIP,53,0.711,7840.0,5.367
STABLE,16,0.710,127.5,32.866
MOMENTUM_DECAY,3,0.720,207.0,39.990



(Do not paste content_hash_id or client_hash_id into the published paper.)


## 7. Artifacts the paper embeds

Three charts and one metrics file. The charts go into `work/figures/`; the metrics
JSON goes into `work/outputs/` and **must be committed** — it is the receipt every
number in the paper traces back to. Dataset CSVs are gitignored and fail CI, so
nothing tabular gets written here.

In [15]:
# FIGURE 1 -- model vs baselines against the base rate
fig, ax = plt.subplots(figsize=(8, 4.5))
xs = np.arange(len(results))
ax.bar(xs - 0.2, results["P@20"], 0.4, label="Precision@20")
ax.bar(xs + 0.2, results["P@50"], 0.4, label="Precision@50")
ax.axhline(te_base, ls="--", lw=1.5, color="black",
           label=f"base rate ({te_base:.1%}) = random ranking")
ax.set_xticks(xs)
ax.set_xticklabels([m.replace(" - ", "\n") for m in results["method"]], fontsize=9)
ax.set_ylabel("Precision")
ax.set_title("Ranking quality vs the base rate (held-out clients)")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig("work/figures/fig1_precision_vs_baserate.png", dpi=150)
plt.close(fig)
print("saved work/figures/fig1_precision_vs_baserate.png")

saved work/figures/fig1_precision_vs_baserate.png


In [16]:
# FIGURE 2 -- feature importance
fig, ax = plt.subplots(figsize=(7, 4.5))
imp = importance.sort_values("importance")
ax.barh(imp["feature"], imp["importance"])
ax.set_xlabel("Random forest feature importance")
ax.set_title("What the model leans on (association, not cause)")
fig.tight_layout()
fig.savefig("work/figures/fig2_feature_importance.png", dpi=150)
plt.close(fig)
print("saved work/figures/fig2_feature_importance.png")

saved work/figures/fig2_feature_importance.png


In [20]:
# FIGURE 3 -- precision as the queue gets longer
ks = [10, 20, 50, 100, 200, 500, 1000]
ks = [k for k in ks if k <= len(d_te)]
fig, ax = plt.subplots(figsize=(7.5, 4.5))
for name, score in [("momentum baseline", d_te["baseline_momentum"]),
                    ("opportunity baseline", d_te["baseline_opportunity"]),
                    ("random forest", d_te["ml_score"])]:
    ax.plot(ks, [precision_at_k(score, y_te, k) for k in ks], marker="o", label=name)
ax.axhline(te_base, ls="--", lw=1.5, color="black", label="base rate")
ax.set_xscale("log")
ax.set_xlabel("Queue length K (log scale)")
ax.set_ylabel("Precision@K")
ax.set_title("Precision decays as the review queue lengthens")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig("work/figures/fig3_precision_at_k_curve.png", dpi=150)
plt.close(fig)
print("saved work/figures/fig3_precision_at_k_curve.png")
SOURCE_LABEL = (f"FlyRank/internship-warehouse build v20260703, "
                f"anchor {ANCHOR} ({'sealed' if SEALED_RUN else 'development'} run)")

saved work/figures/fig3_precision_at_k_curve.png


In [22]:
metrics = {
    "random_state": RANDOM_STATE,
    "source": SOURCE_LABEL,
    "label_definition": "gsc_impressions in (end-30, end] < 0.8 x gsc_impressions in (end-60, end-30]",
    "feature_windows": "(end-120, end-30] only; label window never read by a feature",
    "excluded_tables": {"fact_content_query_90d": QUERY_EXCLUSION_NOTE},
    "exclusions": "content items with imp_w1 < 100 or without a full 120-day history",
    "split": "GroupShuffleSplit on client_hash_id, test_size=0.25",
    "n_rows_modelled": int(len(d)),
    "n_rows_test": int(len(d_te)),
    "n_clients_test": int(d_te["client_hash_id"].nunique()),
    "base_rate_overall": round(base_rate, 4),
    "base_rate_test": round(te_base, 4),
    "strict_label_rate": round(strict_rate, 4),
    "share_of_declines_persisted": (round(float(share_persisted), 4)
                                    if share_persisted == share_persisted else None),
    "consolidation_share": (round(float(CONSOLIDATION_SHARE), 4)
                            if CONSOLIDATION_SHARE is not None else None),
    "results": results.to_dict(orient="records"),
    "feature_importance": importance.round(5).to_dict(orient="records"),
    "false_positives_in_top50": int(len(fp)),
}

with open("work/outputs/capstone_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("wrote work/outputs/capstone_metrics.json  <-- COMMIT THIS FILE")
print(json.dumps({k: v for k, v in metrics.items() if k not in ("results", "feature_importance")}, indent=2))

wrote work/outputs/capstone_metrics.json  <-- COMMIT THIS FILE
{
  "random_state": 42,
  "source": "FlyRank/internship-warehouse build v20260703, anchor 2026-03-31 (development run)",
  "label_definition": "gsc_impressions in (end-30, end] < 0.8 x gsc_impressions in (end-60, end-30]",
  "feature_windows": "(end-120, end-30] only; label window never read by a feature",
  "excluded_tables": {
    "fact_content_query_90d": "fact_content_query_90d aggregates a 90-day window that overlaps the 30-day label window (end-30, end]. Joining it would leak outcome-period impressions into the feature set, so it is excluded."
  },
  "exclusions": "content items with imp_w1 < 100 or without a full 120-day history",
  "split": "GroupShuffleSplit on client_hash_id, test_size=0.25",
  "n_rows_modelled": 70557,
  "n_rows_test": 41709,
  "n_clients_test": 7,
  "base_rate_overall": 0.2478,
  "base_rate_test": 0.2525,
  "strict_label_rate": 0.165,
  "share_of_declines_persisted": 0.6659,
  "consolidation_sha

### Abstract, generated from the run

Five sentences, written last and placed first in the paper. Generated from the
metrics above so the numbers cannot drift between notebook and paper. Read it,
then edit the wording by hand — do not edit the numbers.

In [23]:
best = results.loc[results["AUC"].idxmax()]
rf_row = results[results["method"] == "Random forest"].iloc[0]
mom_row = results[results["method"] == "Baseline A - momentum"].iloc[0]
opp_row = results[results["method"] == "Baseline B - opportunity"].iloc[0]

abstract = f"""
Which pages should a content team review first, ranked by the risk that they lose
search visibility next month? Using {metrics['n_rows_modelled']:,} content items from
{SOURCE_LABEL}, features were built from a 90-day observation window and a page was
labelled declining when its impressions in the following 30-day window fell below 80%
of the prior window -- a forward label the features never observe. A random forest was
compared against two transparent baselines on an identical split grouped by client, so
every score reflects performance on clients never seen in training. Against a base rate
of {te_base:.1%}, the model reached Precision@50 of {rf_row['P@50']:.3f}
(lift {rf_row['lift@50']:.2f}x, AUC {rf_row['AUC']:.3f}), versus {mom_row['P@50']:.3f}
for the momentum baseline and {opp_row['P@50']:.3f} for the click-opportunity rule --
evidence that click-opportunity and decline-risk are distinct constructs. The output is
a ranked review queue with reason codes, intended as decision-support for an editor
deciding where to spend limited hours, not as a prediction of any ranking outcome.
""".strip()

print(abstract)
with open("work/outputs/abstract.txt", "w") as f:
    f.write(abstract)
print("\n---\nsaved to work/outputs/abstract.txt")

Which pages should a content team review first, ranked by the risk that they lose
search visibility next month? Using 70,557 content items from
FlyRank/internship-warehouse build v20260703, anchor 2026-03-31 (development run), features were built from a 90-day observation window and a page was
labelled declining when its impressions in the following 30-day window fell below 80%
of the prior window -- a forward label the features never observe. A random forest was
compared against two transparent baselines on an identical split grouped by client, so
every score reflects performance on clients never seen in training. Against a base rate
of 25.2%, the model reached Precision@50 of 0.460
(lift 1.82x, AUC 0.629), versus 0.300
for the momentum baseline and 0.340 for the click-opportunity rule --
evidence that click-opportunity and decline-risk are distinct constructs. The output is
a ranked review queue with reason codes, intended as decision-support for an editor
deciding where to spend lim

---

## ML-12 — Demo outline, social cut, employer summary

### 5-minute demo outline

**0:00-0:40 — The decision, not the model.** A content team can review maybe 20 pages
a week out of tens of thousands. Show the queue first, before any methodology. The
question on screen: *which 20?*

**0:40-1:30 — Why the obvious approach fails.** Show the click-opportunity rule —
impressions x (1 - CTR) — and its lift against the base rate. It scores decline risk
at roughly chance, because it measures a different thing. This is the hook: the
intuitive heuristic isn't just weaker, it's aimed at the wrong target.

**1:30-2:40 — The one design decision that matters.** Draw the four-window diagram.
Explain that the earlier version of this project predicted a trailing trend from the
metrics that produced it, and why that number would have looked better and meant
nothing. Land the point: *the honest setup scores lower and is worth more.*

**2:40-3:40 — Results.** Figure 1. Base rate line first, then the bars. Say the lift
number, not the raw precision, and say what the model leans on (Figure 2).

**3:40-4:30 — What it cannot do.** No causal claim, no semantic understanding, one
forward window. Show a false positive from the error analysis and explain why the
model was fooled.

**4:30-5:00 — The handoff.** The queue with reason codes, the no-go list, and the
sentence an editor actually acts on.

*If you overrun, cut the results section, not the limitations section.*

---

### Social-post cut

> I spent 8 weeks building a model to rank which web pages a content team should
> review first.
>
> The most useful thing I found wasn't the model. It was that my first version was
> quietly cheating.
>
> I was "predicting" whether a page was declining — using metrics measured over the
> same window that defined declining. The model wasn't forecasting anything. It was
> restating the label.
>
> Rebuilding it with a proper forward window dropped my numbers. That drop was the
> real result.
>
> Second finding: the standard click-opportunity heuristic — high impressions, low
> CTR — ranks decline risk at about chance. Click opportunity and decline risk are
> different problems. Most teams treat them as one.
>
> Full write-up + code below. Built on the FlyRank ML Internship dataset.

*Replace the vague phrasing with your actual numbers before posting.*

---

### Employer-facing summary (3 sentences)

> I built a decision-support model that ranks web pages by the risk of losing search
> visibility in the following month, trained on a ~79M-row search-performance
> warehouse using DuckDB for out-of-core feature engineering and scikit-learn for
> modelling. Validation was grouped by client so every reported score reflects
> performance on organisations unseen during training, and the label was defined on a
> forward window strictly separated from the feature windows to eliminate the
> circularity in my first iteration. The output is a ranked review queue with
> explainable reason codes, benchmarked against two transparent baselines and reported
> alongside its base rate — including the finding that the industry-standard
> click-opportunity heuristic performs near chance on this task.

---

## Self-check

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Every Precision@K in the paper appears next to the base rate and a lift figure
- [ ] `work/outputs/capstone_metrics.json` is committed (dataset CSVs are not)
- [ ] Committed to my repo under `work/notebooks/` — then submit the repo URL on the card
- [ ] My deployed paper has **all 9 sections** — Abstract at the top, Acknowledgments & data credit (the https://flyrank.ai link) at the bottom
- [ ] **ML-12 done above:** 5-minute demo outline + social-post cut + 3-sentence employer summary
- [ ] `submission/paper_url.txt` contains exactly one line: the deployed paper URL